# Pool-based Bayesian optimization — START (fluorination) example

This notebook mirrors [`pool_bo_beginner.py`](pool_bo_beginner.py) step by step.

**Before you run:** set the Jupyter kernel **working directory** to this folder (`Session_7_Bayesian_Opt`, the one that contains `bo_common.py` and the CSV).

**Python environment:** select a kernel where the lesson **ROBERT** package is installed (for example the conda environment `cheminf`). Some default `python3` kernels ship a different PyPI package named `robert` without `RobertModel`, which breaks the BO loop on the first surrogate fit.

ROBERT writes artifacts under `robert_bo_outputs/` unless you pin a `workdir` in `ROBERT_MODEL_KW`.

**Human in the loop:** when you run the BO loop, the notebook will prompt for measured yields (`input()`). For non-interactive execution (e.g. `jupyter nbconvert --execute`), set environment variable `POOL_BO_NOTEBOOK_USE_PRED_MEAN_FOR_YIELDS=1` before starting the kernel so yields default to the model mean (demo only; not a lab measurement).


In [ ]:
%matplotlib inline

import os
import sys
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

SESSION_DIR = Path.cwd()
assert (SESSION_DIR / "bo_common.py").is_file(), (
    "Kernel cwd must be Session_7_Bayesian_Opt (folder containing bo_common.py).\n"
    f"Current cwd: {SESSION_DIR}"
)
if str(SESSION_DIR) not in sys.path:
    sys.path.insert(0, str(SESSION_DIR))

import bo_common


In [ ]:
# ------------------------------- User settings ------------------------------- #
CSV_FILE = "Pd_fluorination_optimization_START_example.csv"
TARGET_COL = "yield"
FEATURE_COLS = ("Pd_precatalyst", "Ligand", "Pd_loading", "Temperature", "Selectfluor")
CAT_COLS = ("Pd_precatalyst", "Ligand")
NUM_COLS = ("Pd_loading", "Temperature", "Selectfluor")
ROBERT_MODEL_KW: dict[str, Any] = {}
FULL_CSV_INSPECTION = False

RANDOM_SEED = 7
N_BO_ITERATIONS = 8
BATCH_SIZE = 1

ACQUISITION = "ei"

TEMPERATURE_SCAN_C = np.arange(40, 101, 10)
PD_LOADING_MOLPCT_SCAN = np.array([5, 10, 15])
SELECTFLUOR_EQUIV_SCAN = np.array([1.0, 1.5, 2.0, 2.5])


def reveal_yield_for_notebook(row, chosen_features, batch_rank, step):
    # Interactive: manual entry. For nbconvert: set POOL_BO_NOTEBOOK_USE_PRED_MEAN_FOR_YIELDS=1.
    if os.environ.get("POOL_BO_NOTEBOOK_USE_PRED_MEAN_FOR_YIELDS", "").lower() in (
        "1",
        "true",
        "yes",
    ):
        return float(row["pred_mean"])
    return bo_common.reveal_yield_manual_from_prediction(
        row, chosen_features, batch_rank, step
    )


## STEP 1 — Load data

In [ ]:
bo_common.print_banner("STEP 1 - LOAD DATA")
df = pd.read_csv(CSV_FILE)
print(f"Loaded CSV from: {CSV_FILE}")
bo_common.quick_inspect_csv(df, target_col=TARGET_COL)
if FULL_CSV_INSPECTION:
    bo_common.print_full_dataframe_inspection(df)


## STEP 2 — Feature columns

In [ ]:
bo_common.print_banner("STEP 2 - DEFINE FEATURE COLUMNS")
feature_cols = list(FEATURE_COLS)
cat_cols = list(CAT_COLS)
num_cols = list(NUM_COLS)
bo_common.validate_bo_feature_columns(df, TARGET_COL, feature_cols, cat_cols, num_cols)
print(f"Target column: {TARGET_COL}")
print(f"Feature columns: {feature_cols}")
print(f"Categorical columns: {cat_cols}")
print(f"Numeric columns: {num_cols}")


## STEP 3 — Scanned pool and initial train / pool split

In [ ]:
rs = RANDOM_SEED

bo_common.print_banner("STEP 3 - UNIVERSE AND INITIAL TRAIN/POOL SPLIT")
_, observed_df, candidate_df = bo_common.build_scanned_pool_with_observed_split(
    df,
    feature_cols,
    cat_cols,
    temperature_scan=TEMPERATURE_SCAN_C,
    pd_loading_scan=PD_LOADING_MOLPCT_SCAN,
    selectfluor_scan=SELECTFLUOR_EQUIV_SCAN,
    random_state=rs,
    verbose=True,
)


## STEP 4 — Surrogate (ROBERT) each iteration

In [ ]:
bo_common.print_banner("STEP 4 - SURROGATE MODEL TRAINING")
print(
    "Each BO iteration runs CURATE→GENERATE→VERIFY→PREDICT on the current training table "
    "(ROBERT only; no separate GPR block)."
)

bo_train = observed_df.copy()
bo_pool = candidate_df.copy()
selected_history: list[dict] = []
score_col = "acq"


## STEP 5 — Sequential BO (setup)

In [ ]:
n_bo_steps = N_BO_ITERATIONS
batch_size = BATCH_SIZE
acquisition_mode = ACQUISITION
acquisition_label = bo_common.start_pool_bo_acquisition_label(acquisition_mode)

bo_common.print_banner("STEP 5 - SEQUENTIAL BO")
print(
    f"ROBERT | acquisition: {acquisition_mode!r} ({acquisition_label}) "
    f"| steps={n_bo_steps} batch={batch_size}"
)
print(
    "Each iteration: rank pool → select batch → plot → append to training and shrink pool."
)


## STEP 5A — Iteration helpers

In [ ]:
bo_common.print_banner("STEP 5A - ITERATION HELPERS")

build_ranking_table = bo_common.make_robert_pool_ranking_fn(
    feature_cols,
    TARGET_COL,
    cat_cols,
    num_cols,
    random_state=rs,
    acquisition_mode=acquisition_mode,
    score_col=score_col,
    ucb_beta=bo_common.START_POOL_BO_UCB_BETA,
    ei_xi=bo_common.START_POOL_BO_EI_XI,
    robert_model_kwargs=ROBERT_MODEL_KW,
)

choose_batch_and_record = bo_common.make_choose_batch_from_ranking_fn(
    feature_cols,
    TARGET_COL,
    score_col,
    reveal_yield_for_notebook,
    "manual",
)


## STEP 5B — BO loop

You will be prompted for a yield each time a condition is selected. For automated runs set `POOL_BO_NOTEBOOK_USE_PRED_MEAN_FOR_YIELDS=1` in the environment (uses `pred_mean` as a stand-in yield).


In [ ]:
bo_common.print_banner("STEP 5B - BO LOOP")
for step in range(1, n_bo_steps + 1):
    if len(bo_pool) == 0:
        print("Pool is empty; stopping early.")
        break

    bo_common.print_banner(f"BO ITERATION {step}")
    ranking, best_so_far = build_ranking_table(bo_train, bo_pool)
    current_batch_size = min(batch_size, len(ranking))
    print(f"Current best observed yield: {best_so_far:.3f}")
    print(f"Batch size this iteration: {current_batch_size}")
    print(f"Top 5 candidates by {acquisition_label}:")
    print(ranking.head(5))

    chosen_records, chosen_keys, history_items, selected_batch = choose_batch_and_record(
        ranking, step, current_batch_size
    )
    for history_item in history_items:
        history_item["best_before"] = best_so_far

    bo_common.plot_iteration_diagnostics(
        ranking=ranking,
        selected_batch=selected_batch,
        feature_cols=feature_cols,
        step=step,
        best_so_far=best_so_far,
        batch_size=current_batch_size,
        score_col=score_col,
        acquisition_label=acquisition_label,
    )
    bo_common.print_iteration_batch_summary(
        ranking, history_items, TARGET_COL, current_batch_size
    )

    selected_history.extend(history_items)
    bo_train, bo_pool = bo_common.update_train_and_pool(
        bo_train, bo_pool, chosen_records, chosen_keys, feature_cols
    )
    print(f"Training rows: {len(bo_train)} | Pool rows: {len(bo_pool)}")


## Wrap-up — Recap and history

In [ ]:
bo_common.print_pool_bo_campaign_recap(acquisition_label, batch_size, mode="manual")

history_df = pd.DataFrame(selected_history)

bo_common.print_bo_history_tables_and_plots(
    history_df,
    TARGET_COL,
    score_col,
    acquisition_label,
    empty_iters_hint=(
        "No BO iterations ran; set a positive N_BO_ITERATIONS in User settings to populate history."
    ),
)

bo_common.print_banner("DONE")
print("Notebook run completed successfully.")
print(
    "Tip: enter measured lab yields for each selected batch member in a human-in-the-loop BO campaign."
)
